In [1]:
import numpy as np
import jax, scipy
from jax import numpy as jnp
from qihd.utils.jax_utils import jax_device

from qihd import MIQPSolver, QIHD, PDQP, MIQP
from qihd.problems import BoxQP, LCQP
from qihd.refiners.jax_adam import JaxAdam

In [2]:
# Set up device; optional if device = 'cpu'
device = "cpu"
jax.config.update("jax_platforms", jax_device(device))

In [3]:
# Generate a random BoxQP instance 

seed = 42
np.random.seed(seed)

n = 50
U = scipy.stats.ortho_group(n, seed).rvs()
eig = np.random.normal(-10, 5, (n))
Q = U.T @ np.diag(eig) @ U
w = np.random.normal(0, 10, (n))
lbs = np.random.uniform(-10, 10, (n))
gaps = np.random.uniform(0.1, 10, (n))
ubs = lbs + gaps
prob = BoxQP(Q, w, bounds=(lbs, ubs))

In [4]:
# Set up backend instance (QIHD)
n_shots = 100
n_steps = 10000
backend = QIHD(n_shots=n_shots, n_steps=n_steps, seed=seed, device=device, slow_a=False)

# Set up refiner instance (JaxAdam)
refiner = JaxAdam(device=device)

# Solve the problem using MIQPSolver
model = MIQPSolver(prob, backend, refiner)
res = model.solve()

# Validate results
xs, cnts = res.refined_samples, res.sample_counts

objs = np.array([prob.obj(x) for x in xs])
maxvios = jax.vmap(lambda x: jnp.max(jnp.concat((lbs - x, x - ubs, jnp.zeros(1)))))(xs)
feas = maxvios < 1e-4
minima = np.min(objs[feas])
minimizer = np.argmin(objs[feas])
minimizer_vios = maxvios[feas][minimizer]
print(f"----- Running QiHD -----\nBackend: {backend.__class__.__name__}; Refiner: {refiner.__class__.__name__}.")
print(f"Incumbent minimum: {minima}, feasibility violations: {minimizer_vios}, success probability : {res.succ_prob()}.")
assert minima < -21736

----- Running QiHD -----
Backend: QIHD; Refiner: JaxAdam.
Incumbent minimum: -21746.948891731714, feasibility violations: 0.0, success probability : 1.0.


## Working with sparse matrices (QUBO)

In [5]:
# Generate sparse problem instances
n = 100
density = 0.1
seed = 42
rng = np.random.default_rng(seed=seed)
def rvs(size=None, random_state=rng):
    return random_state.standard_normal(size)
Q = scipy.sparse.random(n, n, density, random_state=rng, data_rvs=rvs)
Q = Q + Q.T
w = np.random.uniform(-1, 1, (n))

# Random sparse QUBO instance
sparse_prob = MIQP(Q, w, n_binary_vars=len(w))

In [6]:
# Set up backend instance
n_shots = 100
n_steps = 15000
backend = QIHD(n_shots=n_shots, n_steps=n_steps, ballistic=False, slow_a=True, seed=seed, device=device)

# No refiner is needed for QUBO.
# Solve the problem using MIQPSolver
sparse_model = MIQPSolver(sparse_prob, backend)
sparse_res = sparse_model.solve()

# Validate results
objs = np.array([sparse_prob.obj(x) for x in sparse_res.samples])
minima = np.min(objs)
minimizer = np.argmin(objs)
print(f"----- Running QiHD -----\nBackend: {backend.__class__.__name__}.")
print(f"Incumbent minimum: {minima}, success probability : {sparse_res.succ_prob()}.")


----- Running QiHD -----
Backend: QIHD.
Incumbent minimum: -154.62288344065183, success probability : 0.1.
